### 실습

#### 멜론 TOP100 차트 스크래핑

In [ ]:
# 패키지 로드
import pandas as pd
import re
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [ ]:
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 10)
driver.get(f'https://www.melon.com/chart/index.htm')

company_name = wait.until(
    EC.presence_of_element_located((By.ID,'footer'))
    )

In [ ]:
def format_rank_change(rank_text):
    # title 속성이 없는 경우
    if not rank_text:
        return '-'

    rank_text = rank_text.strip()

    # 순위 변동 없음
    if rank_text == '순위 동일':
        return '-'

    # '3단계 상승', '2단계 하락'에서 숫자와 방향 추출
    match = re.search(r'(\d+)\s*단계\s*(상승|하락)', rank_text)

    if match:
        number = match.group(1)
        direction = match.group(2)

        if direction == '상승':
            return f'↑ {number}'

        return f'↓ {number}'

    # 신규 진입 등 예상하지 못한 값은 원문 유지
    return rank_text

In [ ]:
# 차트의 곡 행 가져오기
songs = driver.find_elements(
    By.CSS_SELECTOR,
    '.lst50, .lst100'
)

datas = []

for song in songs:
    rank = song.find_element(By.CSS_SELECTOR, '.rank').text
    title = song.find_element(
        By.CSS_SELECTOR,
        '.rank01 > span > a'
    ).text
    singer = song.find_element(
        By.CSS_SELECTOR,
        '.rank02 a'
    ).text
    album = song.find_element(
        By.CSS_SELECTOR,
        '.rank03 a'
    ).text
    like = song.find_element(By.CLASS_NAME, 'cnt').text

    rank_text = song.find_element(
        By.CSS_SELECTOR,
        '.rank_wrap'
    ).get_attribute('title')

    rank_change = format_rank_change(rank_text)

    img_url = song.find_element(
        By.CSS_SELECTOR,
        '.image_typeAll img'
    ).get_attribute('src')

    datas.append({
        '순위': rank,
        '제목': title,
        '가수': singer,
        '앨범명': album,
        '좋아요': like,
        '랭크변화': rank_change,
        '이미지URL': img_url
    })

df_melon = pd.DataFrame(datas)

display(df_melon)

driver.quit()

df_melon.to_csv(
    '../data/melonTop100.csv',
    encoding='utf-8-sig',
    index=False
)